In [1]:
import pandas as pd
import re

In [ ]:
# LOAD FILES
input_file = "put in your path"
df = pd.read_csv(input_file, encoding="latin1")


In [ ]:
# Master file Pin code or Master city Name
master_file = r"put in your Master file"
master_df = pd.read_excel(master_file, sheet_name='Sheet1')
master_df.columns = master_df.columns.str.strip() 

In [4]:
# USE FUNCTIONS
def clean_text_proper(text):
    if pd.isna(text) or str(text).strip() == "": return ""
    cleaned = re.sub(r'[-+%,]', '', str(text)).strip()
    return cleaned.title()

In [5]:
def clean_address(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""

    # Make text usable
    text = "".join(ch for ch in str(text).strip() if ch.isprintable())

    # --- REMOVE HTTPS / HTTP LINKS  ---
    text = re.sub(
        r'https?\s*[:]\s*(?:[a-zA-Z0-9\./?=&_-]+\s*)+',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # --- REMOVE GOOGLE DRIVE LINKS ---
    text = re.sub(
        r'drive\.google\.com\s*/\s*drive\s*/\s*u\s*/\s*\d+\s*/\s*folders\s*/\s*[A-Za-z0-9\-_]+(\?\S*)?',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # --- REMOVE "Drive Link :- something" ---
    text = re.sub(
        r'Drive\s*Link\s*[:-].*',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # --- REMOVE UNWANTED ADDRESS WORDS ---
    unwanted = ["Aadhar Address", "address", "Rent agreement Address", "agreement Address", 
        "AGREEMENT", "Aadhar", "AADHAR", "Aadhar", "Aadhaar", "AADHAAR",
        "DL-", "CARD", "card", "Adhar-", "Adhar No.", "DL", "DL", "Driving License", 
        "Driving Licence", "Driving Lic", "ADD","Driving Lc", "Licence", "License", 
        "Address", "Address", "Permanent Address", "Present Address", 
        "CORRESPONDENCE ADDRESS", "CORRESPONDENCE", "PERMANENT:",":"]

    for word in unwanted:
        text = re.sub(word, '', text, flags=re.IGNORECASE)

    # Final cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    return text.title()



In [6]:
# PIN CODE extraction
def extract_pin(text):
    if pd.isna(text): return ""
    match = re.search(r'\d{6}', str(text))
    return match.group(0) if match else ""

In [7]:
#Name Format for (First_name,Middle_name and Last_name)
def split_name(name):
    parts = str(name).split()
    if len(parts) == 1: return parts[0], "", ""
    if len(parts) == 2: return parts[0], "", parts[1]
    return parts[0], " ".join(parts[1:-1]), parts[-1]

In [8]:
# Use for Uppercase (name/address)
def format_id(val):
    try:
        if pd.isna(val) or str(val).strip() == "" or str(val).lower() == 'nan':
            return "NA"
        return str(int(float(val)))
    except: return "NA"

In [9]:
# DOB FORMAT

def format_dob(val):
    try:
        if pd.isna(val) or str(val).strip() == "": return ""
        return pd.to_datetime(val).strftime('%Y-%m-%d')
    except: return str(val).replace('/', '-') 

In [10]:
# UNWANTED CHARCTERS REMOVE---
def remove_illegal_chars(val):
    if pd.isna(val): return ""
    return "".join(ch for ch in str(val) if ch.isprintable())

In [11]:
# APPLYING DATA FRAME FOR NAME AND ADDRESS

df['Candidate Name'] = df.iloc[:, 1].apply(clean_text_proper)
df['Father Name'] = df.iloc[:, 2].apply(clean_text_proper)
df['Cleaned_Address'] = df.iloc[:, 4].apply(clean_address)

In [12]:
# UPDATION: WILL EXTRACT THE STUCK PIN CODES FOR ADDRESS

df['PIN_Extracted'] = df['Cleaned_Address'].apply(extract_pin)

In [13]:
# Driver's Name Format
df[['First Name', 'Middle name', 'Last name']] = df['Candidate Name'].apply(lambda x: pd.Series(split_name(x)))

In [14]:
# Pin Code mapping for Master Sheet
df['PIN_Extracted'] = df['PIN_Extracted'].astype(str).str.strip()
master_df['PIN CODE'] = master_df['PIN CODE'].astype(str).str.strip()
master_unique = master_df.drop_duplicates(subset=['PIN CODE'])

In [15]:
# Applying for master sheet
df = df.merge(
    master_unique[['PIN CODE', 'DISTRICT', 'City ID/District ID']], 
    left_on='PIN_Extracted', right_on='PIN CODE', how='left'
)

In [16]:
# FINAL DATAFRAME
final_df = pd.DataFrame()
final_df['UUID'] = df.iloc[:, 0]
final_df['Candidate Name'] = df['Candidate Name']
final_df['First Name'] = df['First Name']
final_df['Middle name'] = df['Middle name']
final_df['Last name'] = df['Last name']
final_df['Father Name'] = df['Father Name']
final_df['DOB'] = df.iloc[:, 3].apply(format_dob)
final_df['DOB Comments'] = ""
final_df['Address'] = df['Cleaned_Address']
final_df['PIN Code'] = df['PIN_Extracted']
final_df['City'] = df['DISTRICT'].fillna('NA')

In [17]:
# NA using for city name and city ID
if 'City ID/District ID' in df.columns:
    final_df['flow_city_id'] = df['City ID/District ID'].apply(format_id)
else:
    final_df['flow_city_id'] = "NA"

In [18]:
# using map function
final_df = final_df.map(remove_illegal_chars)

In [19]:
# OutPut 
output_name = 'Uber_Final_Data.xlsx'
final_df.to_excel(output_name, index=False)

print(f"All data cleaned,")

All data cleaned,
